<a href="https://colab.research.google.com/github/athitthiyan/Learning_Gen_AI/blob/main/task10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install sentence-transformers for semantic search and rank-bm25 for keyword search
!pip install -q sentence-transformers rank-bm25

In [2]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

# Sample corporate/tech dataset
corpus = [
    "How to hard-reset your iPhone 13 if the touch screen is completely frozen or unresponsive.", # Doc 0
    "Troubleshooting guide for iOS updates failing on newer Apple mobile devices.",              # Doc 1
    "The new Samsung Galaxy S26 Ultra features an advanced generative AI camera system.",         # Doc 2
    "Steps to recover a lost Google Pixel account recovery phrase or authentication token.",    # Doc 3
    "Fixing Wi-Fi connectivity drops and network configuration errors on Apple Macbook laptops.", # Doc 4
    "Why is my smartphone battery draining so quickly? Top power optimization tips.",            # Doc 5
]

print(f"Loaded database with {len(corpus)} technical documents.")

# Tokenize corpus for BM25 processing
tokenized_corpus = [doc.lower().split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def sparse_search(query, top_n=5):
    """Executes keyword lookup and returns sorted document indices and raw scores."""
    tokenized_query = query.lower().split(" ")
    scores = bm25.get_scores(tokenized_query)
    # Sort indices by score descending
    top_indices = np.argsort(scores)[::-1][:top_n]

    # Return list of tuples: (doc_id, score)
    return [(idx, scores[idx]) for idx in top_indices if scores[idx] > 0]

# Quick verification test
print("Sparse test search for 'iPhone 13':", sparse_search("iPhone 13"))

# Initialize a lightweight, high-performance bi-encoder model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate dense vector embeddings for the entire corpus
corpus_embeddings = embedding_model.encode(corpus, convert_to_tensor=True)

def dense_search(query, top_n=5):
    """Executes semantic search via cosine similarity and returns sorted indices and scores."""
    query_embedding = embedding_model.encode(query, convert_to_tensor=True)

    # Calculate cosine similarity matrix against all corpus documents
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]

    # Sort top elements
    top_results = np.argsort(cos_scores.cpu().numpy())[::-1][:top_n]

    return [(int(idx), float(cos_scores[idx])) for idx in top_results]

# Quick verification test
print("Dense test search for 'pixels in camera':", dense_search("pixels in camera"))

def reciprocal_rank_fusion(sparse_results, dense_results, k=60, top_n=3):
    """
    Fuses rankings from separate systems using the Reciprocal Rank Fusion formula.
    Inputs are expected to be lists of tuples: (doc_id, score) sorted by relevance.
    """
    rrf_scores = {}

    # Process sparse rankings
    for rank, (doc_id, _) in enumerate(sparse_results):
        # rank starts at 0, so rank position is rank + 1
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + (rank + 1))

    # Process dense rankings
    for rank, (doc_id, _) in enumerate(dense_results):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + (rank + 1))

    # Sort documents based on their combined RRF metrics
    fused_rankings = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)

    return fused_rankings[:top_n]

def hybrid_search_engine(query, top_n=3):
    """Orchestrates both systems and fuses the output."""
    # 1. Gather deep recall arrays from both engines
    sparse_res = sparse_search(query, top_n=10)
    dense_res = dense_search(query, top_n=10)

    # 2. Fuse the rankings using RRF
    hybrid_res = reciprocal_rank_fusion(sparse_res, dense_res, k=60, top_n=top_n)

    # 3. Print pretty outputs for comparison
    print(f"\n==== TARGET QUERY: '{query}' ====")
    for rank, (doc_id, rrf_score) in enumerate(hybrid_res):
        print(f"\n[Rank {rank + 1}] (RRF Score: {rrf_score:.4f})")
        print(f"Document #{doc_id}: {corpus[doc_id]}")

# -----------------------------------------------------------------
# Execution Test 1: The Keyword Trap
# Query uses exact keyword terms from Doc 0, but conceptual theme of Doc 1
hybrid_search_engine("Apple mobile device issues")

# Execution Test 2: Multi-Concept Search
# Query matches semantic context of Doc 5 but specific tech hardware in Doc 4
hybrid_search_engine("Macbook laptop battery optimization")

Loaded database with 6 technical documents.
Sparse test search for 'iPhone 13': [(np.int64(0), np.float64(2.3996477957205307))]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dense test search for 'pixels in camera': [(2, 0.23958879709243774), (3, 0.12979185581207275), (0, 0.1011519730091095), (5, 0.04306134581565857), (1, -0.06324774771928787)]

==== TARGET QUERY: 'Apple mobile device issues' ====

[Rank 1] (RRF Score: 0.0328)
Document #1: Troubleshooting guide for iOS updates failing on newer Apple mobile devices.

[Rank 2] (RRF Score: 0.0318)
Document #4: Fixing Wi-Fi connectivity drops and network configuration errors on Apple Macbook laptops.

[Rank 3] (RRF Score: 0.0161)
Document #5: Why is my smartphone battery draining so quickly? Top power optimization tips.

==== TARGET QUERY: 'Macbook laptop battery optimization' ====

[Rank 1] (RRF Score: 0.0328)
Document #5: Why is my smartphone battery draining so quickly? Top power optimization tips.

[Rank 2] (RRF Score: 0.0323)
Document #4: Fixing Wi-Fi connectivity drops and network configuration errors on Apple Macbook laptops.

[Rank 3] (RRF Score: 0.0159)
Document #2: The new Samsung Galaxy S26 Ultra fe

In [3]:
!pip install scikit-learn

In [5]:
from sklearn.metrics import ndcg_score

evaluation_queries = [
    {
        "query": "iPhone frozen screen",
        "relevant_docs": [0]
    },
    {
        "query": "Apple software update problems",
        "relevant_docs": [1]
    },
    {
        "query": "Samsung AI camera",
        "relevant_docs": [2]
    },
    {
        "query": "battery optimization tips",
        "relevant_docs": [5]
    },
]

def calculate_mrr(results, relevant_docs):
    """
    Computes Reciprocal Rank for a single query.
    """

    for rank, (doc_id, _) in enumerate(results, start=1):
        if doc_id in relevant_docs:
            return 1.0 / rank

    return 0.0

def calculate_ndcg(results, relevant_docs, top_k=5):

    y_true = []
    y_scores = []

    for rank, (doc_id, score) in enumerate(results[:top_k]):

        if doc_id in relevant_docs:
            y_true.append(1)
        else:
            y_true.append(0)

        y_scores.append(score)

    return ndcg_score([y_true], [y_scores])

def evaluate_search_engine():

    mrr_scores = []
    ndcg_scores = []

    for item in evaluation_queries:

        query = item["query"]
        relevant_docs = item["relevant_docs"]

        # Hybrid Retrieval
        sparse_res = sparse_search(query, top_n=10)
        dense_res = dense_search(query, top_n=10)

        hybrid_res = reciprocal_rank_fusion(
            sparse_res,
            dense_res,
            k=60,
            top_n=10
        )

        # MRR
        mrr = calculate_mrr(hybrid_res, relevant_docs)
        mrr_scores.append(mrr)

        # NDCG
        ndcg = calculate_ndcg(hybrid_res, relevant_docs)
        ndcg_scores.append(ndcg)

        print(f"\nQuery: {query}")
        print(f"MRR: {mrr:.4f}")
        print(f"NDCG: {ndcg:.4f}")

    print("\n==========================")
    print(f"Average MRR : {np.mean(mrr_scores):.4f}")
    print(f"Average NDCG: {np.mean(ndcg_scores):.4f}")

In [6]:
evaluate_search_engine()


Query: iPhone frozen screen
MRR: 1.0000
NDCG: 1.0000

Query: Apple software update problems
MRR: 1.0000
NDCG: 1.0000

Query: Samsung AI camera
MRR: 1.0000
NDCG: 1.0000

Query: battery optimization tips
MRR: 1.0000
NDCG: 1.0000

Average MRR : 1.0000
Average NDCG: 1.0000
